In [2]:
# ==============================================================================
# IMPORTS
# ==============================================================================

from openpiv import tools, pyprocess, validation, filters, scaling
import numpy as np
import tifffile as tif
import matplotlib.pyplot as plt
import napari
import pathlib
import scipy
from scipy.interpolate import griddata
from scipy.ndimage import map_coordinates
import pandas as pd

In [3]:
# CROP OFF CORNERS AND VISUALIZE IN NAPARI

first_dataset = tif.imread('/mnt/crunch/Clark/Fly_TFM/data/first/first_best.tif')
sitting_reference = first_dataset[350]
unstressed_reference = first_dataset[-1]

top_left_corner_sitting_reference = sitting_reference[0:830, 0:670]
bottom_left_corner_sitting_reference = sitting_reference[1630:2048, 0:450]
bottom_right_corner_sitting_reference = sitting_reference[1470:2048, 1410:2048]
top_right_corner_sitting_reference = sitting_reference[0:580, 1510:2048]

top_left_corner_unstressed_reference = unstressed_reference[0:830, 0:670]
bottom_left_corner_unstressed_reference = unstressed_reference[1630:2048, 0:450]
bottom_right_corner_unstressed_reference = unstressed_reference[1470:2048, 1410:2048]
top_right_corner_unstressed_reference = unstressed_reference[0:580, 1510:2048]

# ------------------------------------------------------------------------------

# viewer = napari.Viewer()

# # viewer.add_image(top_left_corner_sitting_reference, name='Top Left Corner', colormap='gray', contrast_limits=[0, 255])
# viewer.add_image(bottom_left_corner_sitting_reference, name='Bottom Left Corner', colormap='gray', contrast_limits=[0, 255])
# # viewer.add_image(bottom_right_corner_sitting_reference, name='Bottom Right Corner', colormap='gray', contrast_limits=[0, 255])
# # viewer.add_image(top_right_corner_sitting_reference, name='Top Right Corner', colormap='gray', contrast_limits=[0, 255])
# viewer.add_image(sitting_reference, name='Original Image', colormap='gray', contrast_limits=[0, 255])

# # viewer.add_image(top_left_corner_unstressed_reference, name='Top Left Corner', colormap='gray', contrast_limits=[0, 255])
# viewer.add_image(bottom_left_corner_unstressed_reference, name='Bottom Left Corner', colormap='gray', contrast_limits=[0, 255])
# # viewer.add_image(bottom_right_corner_unstressed_reference, name='Bottom Right Corner', colormap='gray', contrast_limits=[0, 255])
# # viewer.add_image(top_right_corner_unstressed_reference, name='Top Right Corner', colormap='gray', contrast_limits=[0, 255])
# viewer.add_image(unstressed_reference, name='Original Image', colormap='gray', contrast_limits=[0, 255])

# napari.run()

In [ ]:
# %%
# HELPER FUNCTIONS FOR MULTIPASS WARPING
# ------------------------------------------------------------------------------

def compute_dense_field(x, y, u, v, H, W, method='mean'):
    
    grid_x, grid_y = np.meshgrid(np.arange(W), np.arange(H))
    
    if method == 'mean':
        u_dense = np.full((H, W), np.mean(u))
        v_dense = np.full((H, W), np.mean(v))
        
    elif method == 'plane':
        A = np.column_stack([np.ones(x.size), x.ravel(), y.ravel()])
        
        coef_u, *_ = np.linalg.lstsq(A, u.ravel(), rcond=None)
        coef_v, *_ = np.linalg.lstsq(A, v.ravel(), rcond=None)
        
        u_dense = coef_u[0] + coef_u[1]*grid_x + coef_u[2]*grid_y
        v_dense = coef_v[0] + coef_v[1]*grid_x + coef_v[2]*grid_y
        
    elif method == 'rbf':
        from scipy.interpolate import RBFInterpolator
        
        points = np.column_stack([x.ravel(), y.ravel()])
        query_pts = np.column_stack([grid_x.ravel(), grid_y.ravel()])
        
        rbf_u = RBFInterpolator(points, u.ravel(), kernel='thin_plate_spline')
        rbf_v = RBFInterpolator(points, v.ravel(), kernel='thin_plate_spline')
        
        u_dense = rbf_u(query_pts).reshape(H, W)
        v_dense = rbf_v(query_pts).reshape(H, W)
        
    return u_dense, v_dense

def warp_image(img, u_dense, v_dense):
    
    H, W = img.shape
    
    grid_x, grid_y = np.meshgrid(np.arange(W), np.arange(H))
    coords_x = grid_x - u_dense
    coords_y = grid_y + v_dense
    
    return map_coordinates(img, [coords_y, coords_x], order=1)

def compose_fields(u_prev, v_prev, u_new, v_new):
    # total offset at each output point = this pass's offset, plus the
    # previous cumulative offset SAMPLED at the shifted location (not
    # added directly) -- this is the "compose, don't add" logic
    H, W = u_prev.shape
    grid_x, grid_y = np.meshgrid(np.arange(W), np.arange(H))
    
    sample_x = grid_x - u_new
    sample_y = grid_y + v_new
    
    u_prev_sampled = map_coordinates(u_prev, [sample_y, sample_x], order=1)
    v_prev_sampled = map_coordinates(v_prev, [sample_y, sample_x], order=1)
    
    u_total = u_new + u_prev_sampled
    v_total = v_new + v_prev_sampled
    
    return u_total, v_total

In [ ]:
# %%
# MULTIPASS WARPING LOOP
# ------------------------------------------------------------------------------

ref1 = bottom_left_corner_unstressed_reference   # Target, never changes
ref2_original = bottom_left_corner_sitting_reference  # Source, never overwritten

H, W = ref1.shape
u_total = np.zeros((H, W))
v_total = np.zeros((H, W))

# Schedule: one dict per pass -- edit window/search/overlap/method here
pass_schedule = [
    {'window_size': 150, 'search_size': 160, 'overlap': 0,  'dense_method': 'mean'},
    {'window_size': 64,  'search_size': 74,  'overlap': 32, 'dense_method': 'plane'},
    {'window_size': 32,  'search_size': 42,  'overlap': 16, 'dense_method': 'plane'},
]

residual_tol = 1     # px, stop if mean residual drops below this. guess, verify against data
min_window = 12         # px, stop if next pass would go below this. bead-density floor, guess

pass_history = []  # keeps every pass's info for inversion latr

for pass_index, params in enumerate(pass_schedule):

    warped_ref2 = warp_image(ref2_original, u_total, v_total)

    u_new, v_new, sig2noise = pyprocess.extended_search_area_piv(
        warped_ref2.astype(np.int32),
        ref1.astype(np.int32),
        window_size=params['window_size'],
        overlap=params['overlap'],
        dt=1,
        search_area_size=params['search_size'],
        sig2noise_method='peak2peak'
    )

    invalid_mask = validation.sig2noise_val(sig2noise, threshold=1.3)
    
    u_new_filtered, v_new_filtered = filters.replace_outliers(
        u_new, v_new, invalid_mask, method='localmean', max_iter=3, kernel_size=2
    )

    x_new, y_new = pyprocess.get_coordinates(
        image_size=ref1.shape,
        search_area_size=params['search_size'],
        overlap=params['overlap']
    )
    
    x_new_flipped, y_new_flipped, u_new_flipped, v_new_flipped = tools.transform_coordinates(
        x_new, y_new, u_new_filtered, v_new_filtered
    )

    residual_magnitude = np.mean(np.sqrt(u_new_flipped**2 + v_new_flipped**2))

    # interpolate the sparse field to a dense field for composition with the total field
    u_new_dense, v_new_dense = compute_dense_field(
        x_new_flipped, y_new_flipped, u_new_flipped, v_new_flipped,
        H, W, method=params['dense_method']
    )

    u_total, v_total = compose_fields(u_total, v_total, u_new_dense, v_new_dense)

    pass_history.append({
        'pass': pass_index,
        'window_size': params['window_size'],
        'x': x_new_flipped, 'y': y_new_flipped,
        'u': u_new_flipped, 'v': v_new_flipped,
        'residual_magnitude': residual_magnitude,
    })

    print(f"pass {pass_index}: window={params['window_size']}, residual={residual_magnitude:.3f} px")

    next_window = pass_schedule[pass_index+1]['window_size'] if pass_index+1 < len(pass_schedule) else None
    
    if residual_magnitude < residual_tol:
        print("stopped: residual below tolerance")
        break
    if next_window is not None and next_window < min_window:
        print("stopped: next window would drop below bead-density floor")
        break

# Final warped image for visual check
warped_ref2_final = warp_image(ref2_original, u_total, v_total)

viewer = napari.Viewer()
viewer.add_image(ref1, name='Reference 1', colormap='gray', contrast_limits=[0, 255])
viewer.add_image(ref2_original, name='Reference 2 (raw)', colormap='gray', contrast_limits=[0, 255])
viewer.add_image(warped_ref2_final, name='Warped Reference 2 (final)', colormap='gray', contrast_limits=[0, 255])
napari.run()

pass 0: window=150, residual=6.264 px
pass 1: window=64, residual=0.530 px
stopped: residual below tolerance
